# 🧬 Breast Cancer Biomarker Classifier
**Author:** Bhavya Sevak  
**Dataset:** Wisconsin Breast Cancer Dataset (sklearn built-in)  
**Goal:** Classify tumors as malignant or benign using machine learning

---
## Project Overview
This notebook builds and evaluates a machine learning pipeline to classify breast cancer tumors as **malignant** or **benign** based on 30 cell nucleus features extracted from digitized fine needle aspirate (FNA) images.

### Pipeline
1. Data Loading & Exploration
2. Exploratory Data Analysis (EDA)
3. Preprocessing & Feature Engineering
4. Model Training (Logistic Regression, Random Forest, SVM)
5. Model Evaluation & Comparison
6. Feature Importance & Biomarker Identification
7. Conclusions

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')
PALETTE = ['#2ecc71', '#e74c3c']

print('✅ Libraries loaded successfully')

## 2. Load & Explore the Dataset

In [ ]:
# Load dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['diagnosis'] = df['target'].map({1: 'Benign', 0: 'Malignant'})

print('Dataset Shape:', df.shape)
print('\nClass Distribution:')
print(df['diagnosis'].value_counts())
print(f'\nMalignant: {(df.target==0).sum()} ({(df.target==0).mean()*100:.1f}%)')
print(f'Benign:    {(df.target==1).sum()} ({(df.target==1).mean()*100:.1f}%)')
df.head()

In [ ]:
# Basic statistics
print('Basic Statistics (first 5 features):')
df[list(data.feature_names[:5]) + ['diagnosis']].describe().round(2)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df['diagnosis'].value_counts()
axes[0].pie(counts, labels=counts.index, colors=PALETTE[::-1],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')

# Top features by mean difference
mean_diff = abs(df[df.target==1][data.feature_names].mean() -
                df[df.target==0][data.feature_names].mean())
top10 = mean_diff.nlargest(10)
axes[1].barh(top10.index, top10.values, color='#3498db', edgecolor='white')
axes[1].set_title('Top 10 Features by Mean Difference', fontsize=14, fontweight='bold')
axes[1].set_xlabel('|Mean Benign - Mean Malignant|')

plt.tight_layout()
plt.savefig('plots/class_distribution.png', bbox_inches='tight')
plt.show()
print('Plot saved to plots/')

In [ ]:
import os
os.makedirs('plots', exist_ok=True)

# Boxplots for top 6 features
top6_features = mean_diff.nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feature in enumerate(top6_features):
    sns.boxplot(data=df, x='diagnosis', y=feature,
                palette=PALETTE[::-1], ax=axes[i])
    axes[i].set_title(feature, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Top 6 Discriminative Features: Benign vs Malignant',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/feature_boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (mean features only)
mean_features = [f for f in data.feature_names if 'mean' in f]
fig, ax = plt.subplots(figsize=(12, 9))
corr = df[mean_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True,
            ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix (Mean Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 4. Preprocessing & Train/Test Split

In [ ]:
X = df[data.feature_names]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Features:     {X_train.shape[1]}')
print(f'\nTrain class balance: {y_train.mean():.2f} benign ratio')
print(f'Test  class balance: {y_test.mean():.2f} benign ratio')

## 5. Model Training
We train three models using sklearn Pipelines (scaler + classifier):

In [ ]:
# Define pipelines
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True, random_state=42))
    ])
}

# Train and evaluate with cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print('Cross-validation results (5-fold):')
print('-' * 50)

for name, pipeline in models.items():
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
    results[name] = cv_scores
    print(f'{name:25s}: AUC = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print('\n✅ Cross-validation complete')

## 6. Model Evaluation on Test Set

In [ ]:
# Fit all models on full training set and evaluate on test set
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
test_results = {}

for ax, (name, pipeline) in zip(axes, models.items()):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    test_results[name] = {'accuracy': acc, 'auc': auc,
                          'y_pred': y_pred, 'y_prob': y_prob}

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Malignant', 'Benign'],
                yticklabels=['Malignant', 'Benign'])
    ax.set_title(f'{name}\nAcc: {acc:.3f} | AUC: {auc:.3f}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c']

for (name, res), color in zip(test_results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f"{name} (AUC = {res['auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig('plots/roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Model': list(test_results.keys()),
    'Accuracy': [f"{v['accuracy']:.4f}" for v in test_results.values()],
    'ROC-AUC': [f"{v['auc']:.4f}" for v in test_results.values()],
    'CV AUC Mean': [f"{results[k].mean():.4f}" for k in test_results.keys()],
    'CV AUC Std': [f"{results[k].std():.4f}" for k in test_results.keys()],
})
print('📊 Model Performance Summary')
print('=' * 60)
summary

## 7. Feature Importance & Biomarker Identification

In [ ]:
# Random Forest feature importances
rf_model = models['Random Forest'].named_steps['clf']
importances = pd.Series(rf_model.feature_importances_, index=data.feature_names)
top15 = importances.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors_bar = ['#e74c3c' if i >= 10 else '#3498db' for i in range(len(top15))]
bars = ax.barh(top15.index, top15.values, color=colors_bar, edgecolor='white')
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Top 15 Biomarker Features — Random Forest', fontsize=14, fontweight='bold')

# Add value labels
for bar, val in zip(bars, top15.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Top 5 biomarkers'),
                   Patch(facecolor='#3498db', label='Other important features')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('plots/feature_importance.png', bbox_inches='tight')
plt.show()

print('\n🔬 Top 5 Identified Biomarkers:')
for i, (feat, imp) in enumerate(importances.nlargest(5).items(), 1):
    print(f'  {i}. {feat}: {imp:.4f}')

In [ ]:
# Logistic Regression coefficients (top features)
lr_model = models['Logistic Regression']
lr_model.fit(X_train, y_train)
scaler = lr_model.named_steps['scaler']
lr_clf = lr_model.named_steps['clf']

coef_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Coefficient': lr_clf.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors_coef = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_coef, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coefficient (scaled)', fontsize=12)
ax.set_title('Logistic Regression Coefficients\n(Red = malignant indicator, Green = benign indicator)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/lr_coefficients.png', bbox_inches='tight')
plt.show()

## 8. Best Model — Detailed Report

In [ ]:
# Find best model by AUC
best_name = max(test_results, key=lambda k: test_results[k]['auc'])
best_res = test_results[best_name]

print(f'🏆 Best Model: {best_name}')
print(f'   Accuracy : {best_res["accuracy"]:.4f}')
print(f'   ROC-AUC  : {best_res["auc"]:.4f}')
print()
print('Detailed Classification Report:')
print('=' * 50)
print(classification_report(y_test, best_res['y_pred'],
                             target_names=['Malignant', 'Benign']))

## 9. Conclusions

### Key Findings
- All three models achieved **>95% accuracy** on the breast cancer classification task
- The top identified biomarker features were **worst radius**, **worst perimeter**, and **mean concave points** — consistent with clinical literature on tumor morphology
- Random Forest provided the best interpretability through feature importances, directly supporting biomarker discovery
- Logistic Regression coefficients revealed that **larger, more irregular cell nuclei** are the strongest malignancy indicators

### Relevance to Biomedical Research
This pipeline mirrors real-world workflows in biomarker discovery:
- Feature importance = candidate biomarker ranking
- Cross-validation = robust generalization estimate
- ROC-AUC = clinical discriminative ability

### Future Work
- Apply SHAP values for explainability
- Hyperparameter tuning with GridSearchCV
- Extend to metabolomics/genomics datasets (e.g. TCGA)
- Deploy as a simple web app using Streamlit

---
*Bhavya Sevak | M.S. Biomedical Informatics, Arizona State University*